> 在线讲义：[第 17 章 · 在线 RL：SimpleVLA-RL，只看成没成，也能把 17% 练到 92%](https://wikieden.github.io/vla-from-scratch/lessons/17-online-rl.html) · 本书《VLA 从零到 π0》

# 第 17 章 · 在线 RL：SimpleVLA-RL，只看成没成，也能把 17% 练到 92%

配套讲义：仓库里 `vla-deep-dive/notes/lessons/17-online-rl.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python + NumPy，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 写法 | 跑出来 |
|---|---|---|
| 第 1 步 | `TABLE5` | 48.9 → 96.9；91.0 → 99.1 |
| 第 2 步 | `group_advantages(rewards)` | 3/8 成功：+1.29 / −0.77 |
| 第 3 步 | `p_all_same(p, g)` | p = 0.9、G = 8：0.43 |
| 第 3 步 | `clip_ratio_ceiling(0.10, 0.28)`、`softmax_t([2, 1, 0], 1.6)` | 0.128；0.549 / 0.294 / 0.157 |

数据取自 simplevla-rl-2509.09674.txt 表 5（LIBERO，OpenVLA-OFT 骨干）。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
TABLE5 = {"每任务 1 条示范": (48.9, 96.9), "每任务全部 500 条": (91.0, 99.1)}
LIBERO_LONG_ONE_TRAJ = (17.3, 91.7)

**函数 `group_advantages`**：GRPO 式 8：(R_i − 组平均) / 组标准差（总体标准差）；全同时返回全 0。

In [ ]:
def group_advantages(rewards):
    """GRPO 式 8：(R_i − 组平均) / 组标准差（总体标准差）；全同时返回全 0。"""
    g = len(rewards)
    mean = sum(rewards) / g
    std = math.sqrt(sum((r - mean) ** 2 for r in rewards) / g)
    if std == 0:
        return [0.0] * g
    return [(r - mean) / std for r in rewards]

**函数 `p_all_same`**：G 条独立轨迹全成功或全失败（没有梯度）的概率。

In [ ]:
def p_all_same(p, g):
    """G 条独立轨迹全成功或全失败（没有梯度）的概率。"""
    return p**g + (1 - p) ** g

**函数 `clip_ratio_ceiling`**：一次更新里，一个动作的概率最多涨到多少：p_old × (1 + ε_high)。

In [ ]:
def clip_ratio_ceiling(p_old, eps_high):
    """一次更新里，一个动作的概率最多涨到多少：p_old × (1 + ε_high)。"""
    return p_old * (1 + eps_high)

**函数 `softmax_t`**：温度为 t 的 softmax。

In [ ]:
def softmax_t(logits, t):
    """温度为 t 的 softmax。"""
    m = max(logits)
    e = [math.exp((z - m) / t) for z in logits]
    s = sum(e)
    return [x / s for x in e]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 只做 SFT vs 加 RL（LIBERO 平均）")
for k, (sft, rl) in TABLE5.items():
    print(f"  {k}：{sft}% → {rl}%")
print(f"  LIBERO-Long（1 条示范）：{LIBERO_LONG_ONE_TRAJ[0]}% → {LIBERO_LONG_ONE_TRAJ[1]}%")

In [ ]:
print("\n第二步 组内相对优势（G = 8，成功 3 条）")
adv = group_advantages([1, 1, 1, 0, 0, 0, 0, 0])
print(f"  成功：{adv[0]:+.2f}，失败：{adv[-1]:+.2f}")

In [ ]:
print("\n第三步 全同组概率与两个探索技巧")
for p in (0.1, 0.9):
    print(f"  p = {p}，G = 8：一组白跑的概率 {p_all_same(p, 8):.2f}")
print(f"  概率 0.10 的动作一次最多涨到 {clip_ratio_ceiling(0.10, 0.20):.3f}（上界 1.2）/ "
      f"{clip_ratio_ceiling(0.10, 0.28):.3f}（上界 1.28）")
for t in (1.0, 1.6):
    print(f"  温度 {t}：{[round(x, 3) for x in softmax_t([2, 1, 0], t)]}")

## 3. 代码里才看得到的

1. `group_advantages` 在全同时返回全 0，而不是除以 0 报错；真实实现（veRL）会在分母上加一个很小的数。
2. 这里用总体标准差（除以 G）；`torch.std` 默认除以 G − 1，数会稍大一点，结论不变。
3. 二值奖励"平均分给每个 token"，所以同一条轨迹里所有 token 的优势相同；长轨迹和短轨迹在式 7 里先按长度平均再按组平均。

## 4. 自己改一改

- 用 `random` 模拟 1000 组，验证 `p_all_same` 的公式。
- 把 `softmax_t` 的温度从 0.5 扫到 2.0，画出第三个动作的概率。
- 给 `group_advantages` 换成"成功 = 1、失败 = −1"，看优势有没有变。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 看录像学不会骑车，自己骑、只问到头没；
> 同题八人做，比平均好加分，差的减分；
> 全对全错没得比，扔掉再采；放宽上界、升温多试；
> 只看结果会抄近道，规矩要写进奖励。